# 021 — Training: heteroscedastic (Laplace-beta NLL)

Trains the heteroscedastic counterpart of the base architectures — `unet_nll`, `resunet_nll`, `attention_unet_nll` — using the Round 1 recipe (`fixing.md`): `GroupNormalization` instead of `BatchNormalization` (#1), `Adam(weight_decay=...)` (#2), He init instead of Xavier (#3), and a single unified **Laplace-beta NLL** loss (#10) replacing the previous `gaussian_nll`/`beta_nll` split — one checkpoint per architecture instead of two. `efficientnet_unet_nll` moved to `022_training_efficientnet.ipynb` — it keeps training with both Gaussian NLL variants until Round 2 collapses it the same way. See `020_training.ipynb`'s title cell for the full table of companion notebooks.

Only the **artwork-and-mockups** split is used (see §1).


Make the project root importable so `scripts.*` resolves regardless of the notebook's working directory.


In [ ]:
import sys
from pathlib import Path

project_root = Path().absolute()
if project_root.name == "notebooks":
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

Imports, a fixed global seed, and a GPU sanity check.


In [ ]:
import matplotlib.pyplot as plt
import tensorflow as tf

from scripts.config import settings
from scripts.dataset import (
    build_dataset,
    load_image_pairs,
    mockup_aware_train_val_test_split,
)
from scripts.reproducibility import set_global_seed
from scripts.trainer import get_callbacks
from scripts.trainer_nll import compile_model_nll, get_model_nll
from scripts.visualization import plot_training_curves

set_global_seed()

gpus = tf.config.list_physical_devices("GPU")
print(f"GPUs available: {gpus}")
print(f"TensorFlow version: {tf.__version__}")

## 1. Dataset — artwork-and-mockups split

Same split as `020_training.ipynb` §1 (real artworks grouped and leakage-free; mockup groups split at the pair level) — required so these checkpoints are trained and evaluated under the same conditions as the deterministic ones they are compared against.


In [ ]:
pairs = load_image_pairs(settings.IR_DIR, settings.RGB_DIR)
train_pairs, val_pairs, _ = mockup_aware_train_val_test_split(
    pairs,
    train_ratio=settings.TRAIN_RATIO,
    val_ratio=settings.VAL_RATIO,
    mockup_ids=settings.MOCKUP_ARTWORK_IDS,
    mockup_test_ratio=settings.MOCKUP_TEST_RATIO,
    seed=settings.SEED,
)

train_ds = build_dataset(
    train_pairs,
    batch_size=settings.BATCH_SIZE,
    augment=True,
    shuffle=True,
    seed=settings.SEED,
    crop_size=settings.CROP_SIZE,
)
val_ds = build_dataset(
    val_pairs,
    batch_size=settings.BATCH_SIZE,
    augment=False,
    shuffle=False,
)

print(f"Train: {len(train_pairs)} patches ({len(train_ds)} batches)")
print(f"Val:   {len(val_pairs)} patches ({len(val_ds)} batches)")

## 2. Loss function — Laplace-beta NLL

Each architecture outputs two channels per pixel: `mu` (mean, same role as the deterministic models' single output) and a second channel — still named/clipped as `log_var` in the model code for historical reasons (`settings.NLL_LOG_VAR_MIN`/`MAX`), but now interpreted as a **Laplace log-scale** (`log_b`), not a Gaussian log-variance. The loss (`scripts.losses.laplace_nll_loss`, `fixing.md` #10) is:

```
loss = stop_gradient(b) ** beta * (abs(y_true - mu) / b + log_b)     # b = exp(log_b)
```

the Laplace (L1-weighted-by-scale) counterpart of the previous Gaussian NLL (L2-weighted-by-variance) — consistent with the same L1-over-L2 evidence behind `020_training.ipynb`'s `combined_loss` change (#9). `beta` (`settings.NLL_BETA`, default `0.5`) generalises the Seitzer et al. (2022) beta-reweighting trick from the Gaussian variance to the Laplace scale — a motivated adaptation, not the literally published formula. See `code-review.md` §7.6 and `fixing.md` #10 for the full rationale.

Metrics (`mae`, `ssim`, `psnr`) are computed from the `mu` channel only (`scripts.metrics.Mu*Metric`), so they stay directly comparable to the deterministic architectures' metrics in `030_evaluation.ipynb`.

**Note for downstream evaluation**: converting this `log_b` channel to a true standard deviation uses `sigma = exp(log_b) * sqrt(2)` (`scripts.calibration.laplace_sigma_from_scale`) — **not** the Gaussian `exp(0.5 * log_var)` formula every notebook used before Round 1. `efficientnet_unet_nll` (`022_training_efficientnet.ipynb`) still needs the Gaussian formula until Round 2.


## 3. Train all three NLL architectures

Same loop structure as `020_training.ipynb` §3, using `compile_model_nll`'s new default (`loss_name="laplace_nll"`, `beta=settings.NLL_BETA`) — one loss, one checkpoint tree, no more `gaussian_nll`/`beta_nll` split for these three architectures. Checkpoints go to `models/nll/<arch>/best_model.keras`. Set `EPOCHS = 2` for a quick smoke test before committing to a full run.


In [ ]:
ARCHS = ["unet_nll", "resunet_nll", "attention_unet_nll"]
EPOCHS = settings.EPOCHS  # -- lower for a quick smoke test
LOSS_NAME = "laplace_nll"
MODEL_DIR = settings.MODELS_DIR / "nll"
LOG_DIR = settings.LOGS_DIR / "nll"

histories: dict = {}

for arch in ARCHS:
    print(f"\n{'=' * 60}")
    print(f"  Architecture: {arch}  (loss: {LOSS_NAME})")
    print(f"{'=' * 60}")

    model = get_model_nll(arch)
    model = compile_model_nll(
        model,
        lr=settings.LEARNING_RATE,
        loss_name=LOSS_NAME,
        beta=settings.NLL_BETA,
        weight_decay=settings.WEIGHT_DECAY,
    )
    model.summary(line_length=80)

    callbacks = get_callbacks(arch, log_dir=LOG_DIR, model_dir=MODEL_DIR)

    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=EPOCHS,
        callbacks=callbacks,
        verbose=1,
    )
    histories[arch] = history.history

    best_val_loss = min(history.history["val_loss"])
    print(f"\nBest val_loss ({arch}, {LOSS_NAME}): {best_val_loss:.4f}")


## 4. Training curves


In [ ]:
for arch, history in histories.items():
    plot_training_curves(history, title=f"Training history — {arch} ({LOSS_NAME})")
    plt.show()

## 5. Summary

Checkpoints saved to `models/nll/<arch>/best_model.keras`. Logs written to `logs/nll/<arch>/`.


In [ ]:
for arch in ARCHS:
    ckpt = MODEL_DIR / arch / "best_model.keras"
    status = "found" if ckpt.exists() else "MISSING"
    print(f"{arch:<25}: {status}  ({ckpt})")